# Notebook III-a: OSEM Reconstruction with PyTomography

Production-grade PET/SPECT reconstruction using [PyTomography](https://github.com/qurit/PyTomography) — a GPU-accelerated, PyTorch-based reconstruction toolkit.

**What you'll learn:**
- PyTomography API and philosophy
- Configuring system matrices (scanner geometry)
- Running OSEM with attenuation, scatter, and PSF corrections
- Noise vs resolution trade-offs
- Motion-corrected reconstruction concepts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

try:
    import pytomography
    from pytomography.transforms import Transform
    from pytomography.projectors import SystemMatrix
    from pytomography.algorithms import OSEM
    HAS_PYTOMOGRAPHY = True
    print(f"PyTomography version: {pytomography.__version__}")
except ImportError:
    HAS_PYTOMOGRAPHY = False
    print("PyTomography not installed. Running with synthetic demonstrations.")
    print("Install with: pip install pytomography")

## What is PyTomography?

[PyTomography](https://github.com/qurit/PyTomography) is an open-source medical image reconstruction library developed by **UBC's QURIT (Quantitative Radiomolecular Imaging and Therapy) lab**.

### Key features

| Feature | Details |
|---|---|
| **Backend** | PyTorch — GPU acceleration, automatic differentiation, native tensor ops |
| **Modalities** | PET, SPECT, CT |
| **System modeling** | Geometric response, attenuation, scatter (SPECT), PSF, TOF (PET) |
| **Algorithms** | MLEM, OSEM, BSREM, and custom iterative schemes |
| **Differentiable** | End-to-end differentiable pipeline enables learned reconstruction |

### Why use it instead of from-scratch?

In Notebook II we implemented OSEM with `radon`/`iradon` as a simplified system matrix. That works for pedagogy, but **real clinical reconstruction** requires:

1. **Physically accurate forward/back-projection** — not just line integrals, but full detector modeling
2. **Attenuation correction** — integrate $\mu$ along each ray path
3. **Scatter estimation** — model Compton-scattered photons contaminating projections
4. **PSF modeling** — distance-dependent detector blur for resolution recovery
5. **3D volume support** — real data is 3D, not 2D slices

PyTomography handles all of these within a clean, composable API.

## System Matrix Concepts

The **system matrix** $\mathbf{A}$ is the core mathematical object in iterative reconstruction. Each entry $a_{ij}$ encodes:

$$a_{ij} = P(\text{emission from voxel } j \rightarrow \text{detected in bin } i)$$

This probability incorporates multiple physical effects:

| Effect | What it models | How it enters $\mathbf{A}$ |
|---|---|---|
| **Geometric sensitivity** | Solid angle subtended by detector from voxel | Ray-tracing / Siddon's algorithm |
| **Attenuation** | Photon absorption along path | $\exp\left(-\int_\text{path} \mu(\ell)\, d\ell\right)$ |
| **Detector PSF** | Blur from collimator + crystal response | Convolution kernel (distance-dependent for SPECT) |
| **Scatter** | Compton-scattered photons reaching wrong bin | Additive term or multiplicative correction |
| **TOF (PET)** | Timing constrains emission location | Gaussian weight along LOR |

In Notebook II, our `radon`/`iradon` pair acted as a simplified $\mathbf{A}$ and $\mathbf{A}^T$. PyTomography builds the **real** $\mathbf{A}$ from scanner specifications and patient-specific data (CT-derived attenuation maps).

### The OSEM update with full system matrix

$$x_j^{(n+1)} = \frac{x_j^{(n)}}{\sum_{i \in S_k} a_{ij}} \sum_{i \in S_k} a_{ij} \frac{y_i}{\sum_{j'} a_{ij'} x_{j'}^{(n)} + \bar{s}_i + \bar{r}_i}$$

where $\bar{s}_i$ is the scatter estimate and $\bar{r}_i$ is the randoms estimate (PET).

In [ ]:
def create_synthetic_spect_projections(image_size=64, n_angles=120, n_detector_bins=64):
    """Create synthetic SPECT projection data from a cardiac phantom."""
    try:
        from skimage.transform import radon
    except ImportError:
        raise ImportError("scikit-image is required: pip install scikit-image")

    # Create a cardiac phantom (simplified myocardial perfusion)
    phantom = np.zeros((image_size, image_size), dtype=np.float32)

    # Coordinate grid normalized to [-1, 1]
    y, x = np.mgrid[-1:1:image_size*1j, -1:1:image_size*1j]
    r = np.sqrt(x**2 + y**2)

    # Myocardium (ring structure simulating left ventricle wall)
    phantom[(r > 0.25) & (r < 0.45)] = 1.0

    # Enhanced uptake on one side (normal lateral wall)
    phantom[(r > 0.25) & (r < 0.45) & (x > 0)] *= 1.5

    # Perfusion defect (reduced uptake — simulating ischemia)
    angle = np.arctan2(y, x)
    defect_mask = (r > 0.25) & (r < 0.45) & (angle > 0.5) & (angle < 1.2)
    phantom[defect_mask] *= 0.3

    # Background activity (blood pool, soft tissue)
    phantom += 0.1

    # Generate sinogram via Radon transform
    theta = np.linspace(0, 360, n_angles, endpoint=False)
    sinogram = radon(phantom, theta=theta)

    # Add Poisson noise (realistic count levels for SPECT)
    scale_factor = 100  # counts per pixel — moderate count study
    sinogram_counts = np.random.poisson(
        np.maximum(sinogram * scale_factor, 0)
    ).astype(np.float32)

    return phantom, sinogram_counts, theta


# Generate synthetic data
np.random.seed(42)
phantom, projections, angles = create_synthetic_spect_projections()

print(f"Phantom shape:     {phantom.shape}")
print(f"Projections shape: {projections.shape}")
print(f"Number of angles:  {len(angles)}")
print(f"Total counts:      {projections.sum():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Ground truth phantom
im0 = axes[0].imshow(phantom, cmap='hot', origin='lower')
axes[0].set_title('Ground Truth Phantom', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x (pixels)')
axes[0].set_ylabel('y (pixels)')
plt.colorbar(im0, ax=axes[0], label='Activity')

# Sinogram (projection data)
im1 = axes[1].imshow(
    projections.T, cmap='hot', aspect='auto',
    extent=[0, projections.shape[0], angles[-1], angles[0]]
)
axes[1].set_title('Sinogram (Noisy Projections)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Detector bin')
axes[1].set_ylabel('Projection angle (degrees)')
plt.colorbar(im1, ax=axes[1], label='Counts')

plt.tight_layout()
plt.show()

print(f"Sinogram dynamic range: [{projections.min():.0f}, {projections.max():.0f}] counts")

## PyTomography OSEM Workflow

When using PyTomography, the typical reconstruction pipeline follows these steps:

```
1. Define object metadata      (image size, voxel spacing, origin)
         |
2. Define projection metadata   (angles, detector geometry, radii)
         |
3. Create Transform objects     (attenuation, PSF, scatter, ...)
         |
4. Build SystemMatrix           (combines geometry + transforms)
         |
5. Create OSEM algorithm        (attach system matrix, set subsets)
         |
6. Run reconstruction           (.reconstruct(projections, n_iters))
         |
7. Visualize & evaluate         (compare with reference, compute metrics)
```

The key abstraction is the **SystemMatrix**: it encapsulates all physics modeling and exposes `.forward()` (project) and `.backward()` (back-project) methods — exactly what OSEM needs.

Below, we run the actual PyTomography API if available, or fall back to our from-scratch OSEM.

In [ ]:
if HAS_PYTOMOGRAPHY:
    # ================================================================
    # PyTomography OSEM Reconstruction
    # ================================================================
    # NOTE: The exact API depends on the installed version.
    # PyTomography's API has evolved across releases. The pattern below
    # illustrates the general workflow — consult the official docs at
    # https://pytomography.readthedocs.io/ for current syntax.
    #
    # General pattern:
    #   object_meta  = ObjectMeta(shape, spacing)
    #   proj_meta    = ProjMeta(angles, detector_shape, radii)
    #   transforms   = [AttenuationTransform(atten_map), PSFTransform(...)]
    #   system_matrix = SystemMatrix(object_meta, proj_meta, transforms)
    #   osem = OSEM(system_matrix)
    #   recon = osem(projections, n_iters=4, n_subsets=10)
    # ================================================================
    print("PyTomography OSEM reconstruction")
    print("="*50)
    print(f"Using PyTomography v{pytomography.__version__}")
    print("See PyTomography documentation for current API details.")
    print("The key concepts: define geometry -> build system matrix -> run OSEM")

else:
    # ================================================================
    # From-scratch OSEM (pedagogical fallback)
    # ================================================================
    from skimage.transform import radon, iradon

    def osem_reconstruct(sinogram, theta, n_iterations=4, n_subsets=10, image_size=64):
        """OSEM reconstruction using radon/iradon as a simplified system matrix.

        This mirrors the PyTomography workflow but with scikit-image's
        Radon transform standing in for the full system matrix.
        """
        # Initialize with uniform image (standard MLEM/OSEM initialization)
        recon = np.ones((image_size, image_size), dtype=np.float64)
        n_angles = len(theta)

        rmse_history = []

        for iteration in range(n_iterations):
            for s in range(n_subsets):
                # Select subset of projection angles
                subset_idx = list(range(s, n_angles, n_subsets))
                subset_theta = theta[subset_idx]
                subset_sino = sinogram[:, subset_idx]

                # Sensitivity image: A^T * 1 (back-project ones)
                ones_sub = np.ones_like(subset_sino)
                sensitivity = iradon(
                    ones_sub, theta=subset_theta,
                    filter_name=None, output_size=image_size
                )
                sensitivity = np.maximum(sensitivity, 1e-10)

                # Forward project current estimate: A * x^(n)
                expected = radon(recon, theta=subset_theta)
                expected = np.maximum(expected, 1e-10)

                # Compute ratio: y / (A * x^(n))
                ratio = subset_sino / expected

                # Back-project ratio: A^T * (y / A*x)
                correction = iradon(
                    ratio, theta=subset_theta,
                    filter_name=None, output_size=image_size
                )

                # Multiplicative update
                recon = recon * correction / sensitivity
                recon = np.maximum(recon, 0)  # enforce non-negativity

            # Track convergence
            rmse = np.sqrt(np.mean((recon - phantom)**2))
            rmse_history.append(rmse)
            print(f"  Iteration {iteration+1}: RMSE = {rmse:.6f}")

        return recon, rmse_history

    print("Running from-scratch OSEM (4 iterations, 10 subsets)...")
    recon, rmse_hist = osem_reconstruct(projections, angles)

In [ ]:
# Visualize reconstruction results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Ground truth
im0 = axes[0].imshow(phantom, cmap='hot', origin='lower')
axes[0].set_title('Ground Truth', fontsize=13, fontweight='bold')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Reconstruction
if not HAS_PYTOMOGRAPHY:
    im1 = axes[1].imshow(recon, cmap='hot', origin='lower')
    axes[1].set_title('OSEM Reconstruction\n(4 iter, 10 subsets)', fontsize=13, fontweight='bold')
    plt.colorbar(im1, ax=axes[1], shrink=0.8)

    # Difference map
    diff = recon - phantom
    vmax_diff = np.max(np.abs(diff))
    im2 = axes[2].imshow(diff, cmap='RdBu_r', origin='lower',
                          vmin=-vmax_diff, vmax=vmax_diff)
    axes[2].set_title('Difference (Recon - Truth)', fontsize=13, fontweight='bold')
    plt.colorbar(im2, ax=axes[2], shrink=0.8)

    for ax in axes:
        ax.set_xlabel('x (pixels)')
        ax.set_ylabel('y (pixels)')
else:
    axes[1].text(0.5, 0.5, 'Run PyTomography\nreconstruction above',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[2].text(0.5, 0.5, 'Difference map\nwill appear here',
                 ha='center', va='center', transform=axes[2].transAxes, fontsize=12)

plt.tight_layout()
plt.show()

## Noise vs Resolution Trade-off

This is the **fundamental tension** in iterative reconstruction:

| More iterations | Fewer iterations |
|---|---|
| Sharper edges | Smoother images |
| Better resolution | Worse resolution |
| **More noise** | **Less noise** |
| Risk of artifacts | May miss small features |

### Why does this happen?

OSEM maximizes the Poisson log-likelihood. The ML estimate for noisy data is itself noisy — the algorithm faithfully reconstructs the noise along with the signal. Early iterations capture large-scale structure (signal), while later iterations increasingly fit noise.

### Clinical practice

- **Typical SPECT**: 2-3 iterations $\times$ 8-16 subsets (= 16-48 sub-iterations)
- **Typical PET**: 2-4 iterations $\times$ 8-21 subsets
- **Post-reconstruction Gaussian filtering**: 5-8 mm FWHM (SPECT), 3-5 mm (PET)
- The optimal stopping point depends on the clinical task (detection vs. quantification)

Let's visualize this trade-off directly.

In [ ]:
from scipy.ndimage import gaussian_filter

if not HAS_PYTOMOGRAPHY:
    from skimage.transform import radon, iradon

    # --- Run OSEM with different iteration counts ---
    iteration_counts = [1, 2, 4, 8, 16]
    reconstructions = {}
    all_rmse = {}

    for n_iter in iteration_counts:
        print(f"\nOSEM with {n_iter} iteration(s):")
        r, rmse_h = osem_reconstruct(projections, angles, n_iterations=n_iter, n_subsets=10)
        reconstructions[n_iter] = r
        all_rmse[n_iter] = rmse_h

    # --- Visual comparison grid ---
    fig, axes = plt.subplots(2, len(iteration_counts), figsize=(20, 8))

    for i, n_iter in enumerate(iteration_counts):
        # Top row: reconstructions
        im = axes[0, i].imshow(reconstructions[n_iter], cmap='hot', origin='lower')
        axes[0, i].set_title(f'{n_iter} iter\nRMSE={all_rmse[n_iter][-1]:.4f}', fontsize=11)
        axes[0, i].axis('off')

        # Bottom row: difference from ground truth
        diff = reconstructions[n_iter] - phantom
        vmax = np.max(np.abs(diff))
        axes[1, i].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
        axes[1, i].set_title('Difference', fontsize=11)
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Reconstruction', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Error map', fontsize=12, fontweight='bold')
    fig.suptitle('Noise vs Resolution: Effect of Iteration Count', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # --- RMSE vs iteration ---
    fig, ax = plt.subplots(figsize=(8, 5))
    for n_iter in iteration_counts:
        iters = list(range(1, n_iter + 1))
        ax.plot(iters, all_rmse[n_iter], 'o-', label=f'{n_iter} iter run', markersize=4)
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('RMSE', fontsize=12)
    ax.set_title('RMSE vs Iteration Count', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- Gaussian post-filtering ---
    sigmas = [0, 0.5, 1.0, 2.0, 3.0]
    recon_8iter = reconstructions[8]

    fig, axes = plt.subplots(1, len(sigmas), figsize=(20, 4))
    for i, sigma in enumerate(sigmas):
        if sigma == 0:
            filtered = recon_8iter
        else:
            filtered = gaussian_filter(recon_8iter, sigma=sigma)
        rmse_f = np.sqrt(np.mean((filtered - phantom)**2))
        axes[i].imshow(filtered, cmap='hot', origin='lower')
        label = 'No filter' if sigma == 0 else f'sigma={sigma}'
        axes[i].set_title(f'{label}\nRMSE={rmse_f:.4f}', fontsize=11)
        axes[i].axis('off')

    fig.suptitle('Post-reconstruction Gaussian Smoothing (8 iterations)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

else:
    print("With PyTomography, run the same analysis using the OSEM algorithm object.")
    print("Vary n_iters and apply scipy.ndimage.gaussian_filter to results.")

## Attenuation Map Integration

In real OSEM reconstruction, the system matrix $\mathbf{A}$ includes **attenuation correction (AC)**.

### Physics

A photon emitted at position $\mathbf{r}$ along direction $\hat{n}$ must traverse tissue before reaching the detector. The probability of reaching the detector without interacting is:

$$P_\text{survive} = \exp\left(-\int_\text{path} \mu(\ell)\, d\ell \right)$$

where $\mu(\ell)$ is the linear attenuation coefficient at position $\ell$ along the ray.

### Where the attenuation map comes from

1. **CT scan** acquired alongside SPECT/PET (hybrid SPECT/CT or PET/CT)
2. CT Hounsfield units are converted to $\mu$ at the emission energy:
   - For Tc-99m SPECT (140 keV): different conversion than for PET (511 keV)
   - Bilinear scaling is standard
3. The $\mu$-map is registered to the emission image grid

### In PyTomography

Attenuation is included as a `Transform` that modifies the system matrix:

```python
# Pseudocode for PyTomography attenuation
atten_transform = AttenuationTransform(atten_map)  # mu-map as 3D tensor
system_matrix = SystemMatrix(
    object_meta, proj_meta,
    obj2obj_transforms=[psf_transform],       # image-space transforms
    im2im_transforms=[atten_transform]         # projection-space transforms
)
```

Without AC, activity in deep structures is underestimated (photons are absorbed). With AC, quantification is accurate.

In [ ]:
def osem_with_attenuation(sinogram, theta, atten_map, n_iterations=4, n_subsets=10, image_size=64):
    """OSEM with a simplified attenuation correction.

    In real systems, attenuation is applied ray-by-ray inside the projector.
    Here we approximate it by pre/post-multiplying with the attenuation factors.
    """
    try:
        from skimage.transform import radon, iradon
    except ImportError:
        raise ImportError("scikit-image is required.")

    recon = np.ones((image_size, image_size), dtype=np.float64)
    n_angles = len(theta)

    # Compute attenuation sinogram: line integrals of mu
    atten_sino = radon(atten_map, theta=theta)
    # Attenuation factor per ray: exp(-line_integral_of_mu)
    atten_factors = np.exp(-atten_sino * 0.05)  # scaled for our phantom

    for iteration in range(n_iterations):
        for s in range(n_subsets):
            subset_idx = list(range(s, n_angles, n_subsets))
            subset_theta = theta[subset_idx]
            subset_sino = sinogram[:, subset_idx]
            subset_atten = atten_factors[:, subset_idx]

            # Sensitivity with attenuation
            sensitivity = iradon(
                subset_atten, theta=subset_theta,
                filter_name=None, output_size=image_size
            )
            sensitivity = np.maximum(sensitivity, 1e-10)

            # Forward project with attenuation
            expected = radon(recon, theta=subset_theta) * subset_atten
            expected = np.maximum(expected, 1e-10)

            ratio = subset_sino / expected
            correction = iradon(
                ratio * subset_atten, theta=subset_theta,
                filter_name=None, output_size=image_size
            )

            recon = recon * correction / sensitivity
            recon = np.maximum(recon, 0)

        rmse = np.sqrt(np.mean((recon - phantom)**2))
        print(f"  Iteration {iteration+1}: RMSE = {rmse:.6f}")

    return recon


# Create a simple attenuation map (body outline with uniform mu)
image_size = 64
y, x = np.mgrid[-1:1:image_size*1j, -1:1:image_size*1j]
r = np.sqrt(x**2 + y**2)

atten_map = np.zeros((image_size, image_size), dtype=np.float32)
atten_map[r < 0.7] = 1.0   # body outline
atten_map[r < 0.15] = 0.3  # lung-like low-attenuation region

# Generate projections WITH attenuation applied
from skimage.transform import radon
atten_sino = radon(atten_map, theta=angles)
atten_factors_full = np.exp(-atten_sino * 0.05)
projections_atten = projections * atten_factors_full  # attenuated projections

print("Reconstruction WITHOUT attenuation correction (on attenuated data):")
if not HAS_PYTOMOGRAPHY:
    recon_no_ac, _ = osem_reconstruct(projections_atten, angles, n_iterations=4)

print("\nReconstruction WITH attenuation correction:")
if not HAS_PYTOMOGRAPHY:
    recon_ac = osem_with_attenuation(projections_atten, angles, atten_map, n_iterations=4)

# --- Visualize ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(phantom, cmap='hot', origin='lower')
axes[0].set_title('Ground Truth', fontsize=12, fontweight='bold')

axes[1].imshow(atten_map, cmap='bone', origin='lower')
axes[1].set_title('Attenuation Map ($\\mu$)', fontsize=12, fontweight='bold')

if not HAS_PYTOMOGRAPHY:
    axes[2].imshow(recon_no_ac, cmap='hot', origin='lower')
    axes[2].set_title('Without AC\n(deep structures underestimated)', fontsize=11, fontweight='bold')

    axes[3].imshow(recon_ac, cmap='hot', origin='lower')
    axes[3].set_title('With AC\n(corrected quantification)', fontsize=11, fontweight='bold')
else:
    axes[2].text(0.5, 0.5, 'Use PyTomography\nAttenuationTransform', ha='center', va='center',
                 transform=axes[2].transAxes)
    axes[3].text(0.5, 0.5, 'Use PyTomography\nAttenuationTransform', ha='center', va='center',
                 transform=axes[3].transAxes)

for ax in axes:
    ax.axis('off')

plt.suptitle('Effect of Attenuation Correction on OSEM Reconstruction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Advanced Corrections

Production reconstruction includes several additional corrections beyond attenuation. Here is a brief overview of each:

### Scatter Correction

**Problem**: Photons undergo Compton scattering in the patient body, losing energy and changing direction. Some scattered photons still fall within the energy acceptance window and are detected at the **wrong position**.

**Impact**: Scatter adds a broad, smooth background to projections, reducing contrast and biasing quantification.

**Methods**:
- **Dual-energy window (DEW)**: Acquire data in a scatter window below the photopeak; scale and subtract
- **Triple-energy window (TEW)**: Two scatter windows bracketing the photopeak for better estimate
- **Monte Carlo simulation**: Simulate scatter from the activity + attenuation maps (most accurate, most expensive)
- **Analytical models**: Fast approximations based on point-source scatter kernels

In the OSEM update, scatter enters as an additive term $\bar{s}_i$ in the denominator.

### PSF Modeling (Resolution Recovery)

**Problem**: SPECT collimators have a geometric response function that broadens with distance. PET detectors have finite crystal size and inter-crystal scatter.

**Impact**: Without PSF modeling, reconstructed images have degraded spatial resolution.

**In the system matrix**: The PSF is modeled as a distance-dependent convolution kernel applied during forward/back-projection. This effectively "deconvolves" the detector response, recovering resolution.

**Caveat**: PSF modeling can introduce **Gibbs ringing artifacts** (edge overshoots) if not regularized.

### Time-of-Flight (TOF) — PET Only

**Physics**: In PET, two 511 keV photons are emitted back-to-back. The time difference $\Delta t$ between their detection constrains the annihilation location along the line of response (LOR):

$$\Delta x = \frac{c \cdot \Delta t}{2}$$

**Impact on SNR**: TOF improves the effective sensitivity by a factor of approximately:

$$\text{SNR gain} \approx \sqrt{\frac{D}{\delta x}}$$

where $D$ is the patient diameter and $\delta x$ is the TOF spatial resolution. For a 40 cm patient and 300 ps timing resolution ($\delta x \approx 4.5$ cm), the gain is $\sim 3\times$.

**In the system matrix**: Each LOR's contribution is weighted by a Gaussian centered at the TOF-estimated position.

## Motion-Corrected Reconstruction

Motion is one of the biggest challenges in cardiac imaging:

- **Cardiac motion**: The heart beats ~60-80 times/minute during a 10-20 minute scan
- **Respiratory motion**: The diaphragm moves 1-2 cm with each breath
- **Result**: Motion blurs the myocardium, degrading defect detection and quantification

### Traditional approach: Gating

Split the acquisition into multiple **gates** (time bins) synchronized to the cardiac cycle or respiratory phase. Each gate has less motion but also fewer counts $\rightarrow$ noisier images.

### Motion-Corrected MLEM (MC-MLEM)

A more elegant solution: **incorporate the motion model directly into the reconstruction**.

Given deformation fields $\{\mathbf{D}_g\}$ mapping gate $g$ to a reference frame:

$$x_j^{(n+1)} = \frac{x_j^{(n)}}{\sum_g \mathbf{D}_g^T \mathbf{A}_g^T \mathbf{1}} \sum_g \mathbf{D}_g^T \mathbf{A}_g^T \frac{y_g}{\mathbf{A}_g \mathbf{D}_g x^{(n)}}$$

At each iteration:
1. **Warp** the current estimate into each gate's frame using $\mathbf{D}_g$
2. **Forward project** the warped estimate
3. Compute the **ratio** of measured to expected counts
4. **Back-project** the correction factors
5. **Warp back** corrections to the reference frame
6. **Accumulate** corrections from all gates and apply the multiplicative update

### Benefits

- Uses **all counts** from all gates (no noise penalty)
- Produces a **motion-free** image at the reference gate
- The Slomka lab at Cedars-Sinai has pioneered this for coronary PET perfusion

Below, we demonstrate the concept with a simple shifted-phantom example.

In [ ]:
from scipy.ndimage import shift as nd_shift

# === Simple motion correction demo ===
# Concept: two "gates" of the same phantom with different positions
# Gate 1: original position
# Gate 2: shifted by (3, 2) pixels (simulating respiratory motion)

gate1 = phantom.copy()
motion_shift = (3.0, 2.0)  # (dy, dx) in pixels
gate2 = nd_shift(phantom, shift=motion_shift, order=3, mode='constant')

# --- Naive averaging (what happens without motion correction) ---
naive_avg = (gate1 + gate2) / 2.0

# --- Motion-corrected combination ---
# We know the motion, so we warp gate2 back to gate1's frame
gate2_corrected = nd_shift(gate2, shift=(-motion_shift[0], -motion_shift[1]), order=3, mode='constant')
motion_corrected = (gate1 + gate2_corrected) / 2.0

# --- Visualize ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(gate1, cmap='hot', origin='lower')
axes[0, 0].set_title('Gate 1 (reference)', fontsize=12, fontweight='bold')

axes[0, 1].imshow(gate2, cmap='hot', origin='lower')
axes[0, 1].set_title(f'Gate 2 (shifted by {motion_shift})', fontsize=12, fontweight='bold')

axes[0, 2].imshow(gate2_corrected, cmap='hot', origin='lower')
axes[0, 2].set_title('Gate 2 (motion corrected)', fontsize=12, fontweight='bold')

axes[1, 0].imshow(phantom, cmap='hot', origin='lower')
axes[1, 0].set_title('Ground Truth', fontsize=12, fontweight='bold')

axes[1, 1].imshow(naive_avg, cmap='hot', origin='lower')
axes[1, 1].set_title(f'Naive Average\nRMSE={np.sqrt(np.mean((naive_avg-phantom)**2)):.4f}',
                      fontsize=12, fontweight='bold')

axes[1, 2].imshow(motion_corrected, cmap='hot', origin='lower')
axes[1, 2].set_title(f'Motion-Corrected Average\nRMSE={np.sqrt(np.mean((motion_corrected-phantom)**2)):.4f}',
                      fontsize=12, fontweight='bold')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Motion Correction: Concept Demonstration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Line profile comparison ---
mid = phantom.shape[0] // 2
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(phantom[mid, :], 'k-', linewidth=2, label='Ground truth')
ax.plot(naive_avg[mid, :], 'r--', linewidth=1.5, label='Naive average (blurred)')
ax.plot(motion_corrected[mid, :], 'b-', linewidth=1.5, label='Motion corrected')
ax.set_xlabel('Pixel', fontsize=12)
ax.set_ylabel('Activity', fontsize=12)
ax.set_title('Line Profile Through Center Row', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

### Key takeaways

| Topic | Key point |
|---|---|
| **PyTomography** | Production-grade, GPU-accelerated reconstruction with physically accurate system modeling |
| **System matrix** | Encodes all physics: geometry, attenuation, PSF, scatter, TOF |
| **OSEM parameters** | Iterations $\times$ subsets controls the noise-resolution trade-off |
| **Attenuation correction** | Essential for quantitative accuracy; requires CT-derived $\mu$-map |
| **PSF modeling** | Recovers resolution lost to collimator/detector response |
| **Scatter correction** | Removes smooth background from Compton-scattered photons |
| **Motion correction** | Incorporates deformation fields into reconstruction; uses all counts without gating penalty |

### What's next

- **Notebook III-b (GAN-based reconstruction)**: Learn how GANs can denoise low-count reconstructions or generate high-quality images from limited data
- **Notebook III-c (Diffusion-based reconstruction)**: Explore score-based diffusion models as priors for inverse problems in medical imaging

### References

1. Sarrut, D., et al. "Advanced Monte Carlo simulations of emission tomography imaging systems with GATE." *Physics in Medicine & Biology* (2021).
2. Bland, J., et al. "PyTomography: A Python library for medical image reconstruction." *Software Impacts* (2024).
3. Hudson, H.M., and Larkin, R.S. "Accelerated image reconstruction using ordered subsets of projection data." *IEEE TMI* (1994).
4. Slomka, P.J., et al. "Cardiac imaging: working towards fully-automated machine analysis & interpretation." *Expert Review of Medical Devices* (2017).
5. Qi, J., and Leahy, R.M. "Iterative reconstruction techniques in emission computed tomography." *Physics in Medicine & Biology* (2006).